# 0. Environment Setup

Mounts Google Drive, makes the shared modules in src/ importable, seeds the random number generators and creates the project folder structure.

After editing a module in src/, call reload_project_modules() and re-run the cells that import from it.

In [1]:
# =====================================================
# 0.1 - BOOTSTRAP: DRIVE MOUNT AND MODULE PATH
# =====================================================
import importlib
import sys
from pathlib import Path

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/VineProject_ToF")
else:
    PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "src" / "config.py").exists():
    raise FileNotFoundError(f"config.py not found under: {PROJECT_ROOT / 'src'}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import config


def reload_project_modules() -> None:
    """Reload the modules in src/ after editing them.

    Python caches imported modules, so a module edited on Drive is not picked
    up by a simple re-import. Run this cell after uploading a new version of
    config.py or data_preparation.py, then re-run the cells that import from
    them.
    """
    # config first: the other modules read its constants at import time.
    for module_name in ("config", "data_preparation"):
        if module_name in sys.modules:
            importlib.reload(sys.modules[module_name])
            print(f"  reloaded: {module_name}")


config.print_environment_info()
config.set_seed()
config.create_project_structure()

Mounted at /content/drive
ENVIRONMENT INFO
Platform:       colab
Base path:      /content/drive/MyDrive/VineProject_ToF
Device:         cpu
PyTorch:        2.11.0+cpu
OpenCV:         5.0.0
Albumentations: 2.0.8
NumPy:          2.0.2
[OK] Random seed set to 42 (cuDNN deterministic: False)
Project structure ready at: /content/drive/MyDrive/VineProject_ToF
All folders already existed.


# 1. Label Map Generation

Converts the LabelMe JSON annotations into segmentation masks: a multiclass label map (5 classes) and a binary cordon/background map, plus their colored versions for visual inspection.

Label aliases, drawing priorities and ignored labels are defined in config.py, section 5. Set overwrite=True to rebuild everything from scratch.

Input: data/raw/annotations/*.json — Output: data/processed/labels/ and data/processed/colored_masks/

In [2]:
# =====================================================
# 1.1 - GENERATE LABEL MAPS FROM LABELME ANNOTATIONS
# =====================================================
from data_preparation import generate_label_maps

generate_label_maps(overwrite=False)

[OK] Label maps already exist — skipping generation.
     Call generate_label_maps(overwrite=True) to rebuild them.


# 2. Skeleton Map Generation

Precomputes the tubed skeleton of the elongated classes (Cane, Cordon), used by the clDice metric and by the skeleton recall loss. Running it offline keeps Zhang-Suen thinning out of the training loop, where it would dominate data loading time.

The cordon is skeletonised from the unified binary mask, so that canes crossing in front of it do not break the centreline; the occluded pixels are then removed from the result.

Skeleton classes and dilation kernel are set in config.SKELETON_LOSS_CONFIG.

Input: data/processed/labels/ — Output: data/processed/skeleton/ and data/processed/colored_masks/skeleton_overlay/

In [3]:
# =====================================================
# 2.1 - GENERATE TUBED SKELETON MAPS
# =====================================================
from data_preparation import generate_skeleton_maps

generate_skeleton_maps(overwrite=False)

SKELETON MAPS GENERATION
Input labels:     /content/drive/MyDrive/VineProject_ToF/data/processed/labels/multiclass
Files to process:   93
Skeleton classes: ['Cane', 'Cordon']
Dilation kernel:  2x2
Overwrite:        False


Generating skeleton maps: 100%|██████████| 93/93 [01:17<00:00,  1.20it/s]

SKELETON MAPS GENERATION COMPLETED
  Skeleton maps:    /content/drive/MyDrive/VineProject_ToF/data/processed/skeleton
  Colored overlays: /content/drive/MyDrive/VineProject_ToF/data/processed/colored_masks/skeleton_overlay


# 3. Train / Validation / Test Split

Partitions the dataset into three sets. The split is computed on the original file stems, before augmentation, so the augmented copies of one image can never end up in two different partitions.

The result is written to data/splits/splits.json and reloaded by every other notebook via load_splits(), which guarantees EDA, training and evaluation all work on the same partition. Ratios and seed are set in config.py.

Output: data/splits/splits.json

In [5]:
# =====================================================
# 3.1 - CREATE TRAIN / VAL / TEST SPLIT
# =====================================================
from data_preparation import create_splits

SPLITS = create_splits(overwrite=False)

DATASET SPLIT
Total samples: 93
  train     65 ( 69.9%)
  val       14 ( 15.1%)
  test      14 ( 15.1%)
Random seed:   42
Splits hash:   d57f79eb
Saved to:      /content/drive/MyDrive/VineProject_ToF/data/splits/splits.json


# 4. Offline Augmentation

Materialises the train, val and test folders used for training. Each training sample is written together with one flipped copy per augmentation; validation and test samples are copied unchanged, so the three folders share the same layout.

Flips are applied offline because reading from Drive is the bottleneck, and a fixed augmented set makes runs directly comparable. Every modality receives the same geometric transformation, keeping images and label maps aligned.

Input: data/raw/, data/processed/, data/splits/splits.json — Output: data/augmented/

In [6]:
# =====================================================
# 4.1 - BUILD THE AUGMENTED DATASET
# =====================================================
from data_preparation import build_augmented_dataset

build_augmented_dataset(overwrite=False)

OFFLINE AUGMENTATION
Modalities:    ['intensity', 'depth', 'confidence', 'labels_multi', 'labels_binary', 'skeleton']
Augmentations: ['flipH', 'flipV', 'flipHV']
Train:         65 -> 260
Val:           14 (copied unchanged)
Test:          14 (copied unchanged)
Overwrite:     False


Writing test: 100%|██████████| 14/14 [00:19<00:00,  1.39s/it]

AUGMENTATION COMPLETED
  train  expected   260  [OK]
  val    expected    14  [OK]
  test   expected    14  [OK]
